# Le Perceptron

In [4]:
import numpy as np
import scipy 
import matplotlib.pyplot as plt 
import h5py 
from PIL import Image
from scipy import ndimage

## Importation du jeu de données (chats) :

In [5]:
def load_dataset():
    with h5py.File('datasets/train_catvnoncat.h5', "r") as train_dataset:
        train_set_x_orig = np.array(train_dataset["train_set_x"][:])
        train_set_y_orig = np.array(train_dataset["train_set_y"][:])

    with h5py.File('datasets/test_catvnoncat.h5', "r") as test_dataset:
        test_set_x_orig = np.array(test_dataset["test_set_x"][:])
        test_set_y_orig = np.array(test_dataset["test_set_y"][:])
        classes = np.array(test_dataset["list_classes"][:])

    train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))

    return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes 

In [6]:

x_train, y_train, x_test, y_test, classes = load_dataset()

### Affichage des informations sur le jeu de données :

In [7]:
print("Taille du jeu d'entraînement : ",y_train.shape)
print("Taille du jeu de test : ",y_test.shape)
print("Dimensions de l'image : ",x_train[0].shape)
print("Format (shape) du jeu d'entraînement : ",x_train.shape)


Taille du jeu d'entraînement :  (1, 209)
Taille du jeu de test :  (1, 50)
Dimensions de l'image :  (64, 64, 3)
Format (shape) du jeu d'entraînement :  (209, 64, 64, 3)


NB : La taille de l'image est de 64x64 px, mais il existe une troisième dimension de taille 3. 
C'est parce que l'image est composée de trois couches : une couche rouge, une couche bleue et une couche verte (RVB). Chaque valeur de chaque couche est comprise entre 0 et 255.

## Aplatissement des images (Flattening) :
Pour pouvoir fournir des images à notre algorithme, nous devons aplatir chaque image en un vecteur. Nous utiliserons pour cela la fonction `reshape`. \
`reshape` prend la nouvelle forme souhaitée et renvoie le tableau transformé. \
Dans notre cas, la forme initiale est (209, 64, 64, 3),
ce que l'on peut décomposer en 2 parties :
- 209 : le nombre d'images 
- 64, 64, 3 : les dimensions de chaque image de notre jeu de données 
La seconde partie sera celle concernée par l'aplatissement, c'est pourquoi nous utiliserons `reshape` comme suit :
> `flat_x_train = x_train.reshape(x_train.shape[0], -1)`

- Le premier argument donné à reshape est `x_train.shape[0]` (soit 209, le nombre d'images), la partie de la forme initiale que nous voulons conserver.
- Le second argument est `-1`, ce qui indique à la fonction de regrouper tout le reste dans une seule dimension. \
Cela donne à notre nouveau jeu de données la forme suivante :
(209, 12288)

In [8]:
#plt.imshow(x_train[11])
img = x_train[11]
x_train.shape
flat_x_train = x_train.reshape(x_train.shape[0], -1)
flat_x_test = x_test.reshape(x_test.shape[0], -1)

print("flat_x_train.shape", flat_x_train.shape)
print("flat_x_test.shape", flat_x_test.shape)

flat_x_train.shape (209, 12288)
flat_x_test.shape (50, 12288)


## Standardisation :
Nous divisons les valeurs des pixels des images aplaties par la valeur maximale qu'un pixel peut avoir, afin de faire fluctuer leurs valeurs entre 0 et 1.


In [9]:
train_set_x = flat_x_train / 255
test_set_x = flat_x_test / 255

## Fonction d'activation :
Comme notre problème est un problème de classification, nous choisissons la fonction sigmoïde :
$$y=\frac{1}{1+e^{-z}}$$

z est une fonction linéaire :
$$ z = x_1.w_1+x_2.w_2 + b $$
Cette fonction trace une ligne appelée "frontière de décision" (decision boundary). Cette ligne sépare les cas en 2 catégories :
- cas positifs z > 0 
- cas négatifs z < 0 \ 
Nous injectons cette fonction dans une sigmoïde car nous devons traiter des probabilités.
Avec ce réglage :
- quand z tend vers +inf, y tend vers 1
- quand z tend vers -inf, y tend vers 0
- quand z tend vers 0, y tend vers 0,5, ce qui sera notre seuil pour décider entre les cas. \
<img src="img/y.png">


### Définissons-la :

In [10]:
def sigmoid(z):
   return 1 / (1 + np.exp(-z))

def linear_function(X, W, b):
    linear_y_hat = np.dot(X , W) + b
    return linear_y_hat

def init_with_zero(dim):
    W = np.zeros((dim, 1))
    b = 0
    return W, b

## Fonction de Coût (Cost function) :

#### Théorie :
Le but principal de la fonction de coût est de pénaliser notre algorithme lorsque la prédiction $\hat{y}$ est différente du y réel du jeu de données.
- si $y = 0$, le résultat de notre fonction de coût doit être plus grand quand $\hat{y}$ approche 1
- si $y = 1$, le résultat de notre fonction de coût doit être plus grand quand $\hat{y}$ approche 0

Nous utiliserons la propriété de la fonction logarithme népérien :

- si $y = 0$, nous utilisons $$-\log{(1-\hat{y})}$$ (Note: corrigé ici pour la logique). Cette fonction approchera $+\infty$ quand $\hat{y}$ approche 1 et 0 quand $\hat{y}$ approche 0.

- si $y = 1$, nous utilisons $$-\log{(\hat{y})}$$, cette fonction approchera $+\infty$ quand $\hat{y}$ approche 0 et 0 quand $\hat{y}$ approche 1.
#### Démonstration Python :
Veuillez consulter le fichier `./demo/the_cost_evolution.py` pour une démonstration Python des graphiques ci-dessous.
<img src="img/the_cost_evolution.png">

Cette fonction :
$$ -y_i\log{\hat{y}_i}-(1 - y_i)\log{(1 - \hat{y}_i)}$$
est la combinaison des 2 fonctions vues précédemment car :
- si y = 1, cette fonction devient : 
$$-\log{(\hat{y})}$$ 
- si y = 0, elle devient $$-\log{(1-\hat{y})}$$
Lorsque y et $\hat{y}$ sont différents, cette fonction tend vers +$\infty$, lorsqu'ils sont identiques, elle tend vers 0. 
C'est donc le cadre parfait pour maximiser le coût quand y et $\hat{y}$ diffèrent.
Comme la fonction de coût doit être la moyenne de tous les coûts individuels, la fonction devient :
$$J = - \frac{1}{m} \sum_{i = 1}^{m} [y_i\log{\hat{y}_i}+(1 - y_i)\log{(1 - \hat{y}_i)}]$$
- <em>m est la taille du jeu de données</em>
<img src="img/logloss_plot.png">


## Descente de Gradient
L'idée de la descente de gradient est d'itérer une fonction qui mettra à jour nos paramètres à chaque fois jusqu'à obtenir le meilleur modèle correspondant à notre jeu de données.
La fonction de descente de gradient sera presque la même que pour la régression linéaire :
$$ \theta_j = \theta_j - \alpha . \frac{\partial J}{\partial \theta_j} $$
- $j$ est l'indice d'itération sur l'époque (epoch)
- l'époque est le nombre de fois que l'algorithme va itérer 
- $\alpha$ est le taux d'apprentissage (learning rate)
- $\frac{\partial J}{\partial \theta_j}$ : dérivée de la fonction de coût 
<p>
La dérivée nous donne la pente d'une fonction :
- si le signe de la dérivée est négatif, cela signifie que nous avons une pente descendante
- si le signe de la dérivée est positif, cela signifie que nous avons une pente montante
<p>Sachant que la logloss est une fonction convexe (comme on le voit sur le graphique), si nous voulons trouver son minimum, nous devrons avancer si le signe de la dérivée est négatif (pente descendante) et reculer si le signe est positif (pente montante).</p></p>


##### NB : cela doit être fait sur tous nos paramètres $(w_1, w_2 ... w_n, b)$ séparément et simultanément pour chaque époque ; tous les paramètres doivent être mis à jour !!!
Voici ce que nous donne la dérivée de la fonction de coût par rapport à chacun de nos paramètres :

<img src="img/dw_db.png">


### NB : rappelez-vous que notre fonction linéaire est $ z = x_1.w_1+x_2.w_2 + b $

- $a_i$ est la prédiction $\hat{y}$ sur la i-ème valeur de notre jeu de données (résultat du modèle sur le i-ème exemple).
- $m$ est la taille du jeu de données.
- $y_i$ est l'observation réelle sur la i-ème valeur.

Plus d'explications sur les dérivées via ce lien :
https://www.youtube.com/watch?v=GzphoJOVEcE&list=PLkDaE6sCZn6Ec-XTbcX1uRg2_u4xOEky0&index=11

## Vectorisation
La vectorisation est l'étape entre la théorie et la pratique. Nous devons convertir toutes nos équations mathématiques en vecteurs pour optimiser l'exécution.
#### Vectorisation de la fonction linéaire :
<img src="img/lfv.png">

#### Vectorisation de la Sigmoïde :
<img src="img/sv_1.png">
<img src="img/sv_2.png">


#### Vectorisation de la fonction de Coût :
##### NB : les multiplications contenues dans la fonction de coût ne sont pas des multiplications de matrices mais des multiplications simples élément par élément (row-by-row).

<img src="img/cost_fn_vectorisation.png">


#### Vectorisation du Gradient :
##### NB : X doit être transposé 

<img src="img/vectorisation_derivative.png">

##### Ainsi, nos formules pour la vectorisation du Gradient deviennent : 
$$\frac{\partial{L}}{\partial{W}} = \frac{1}{m} X^T . (\hat{Y} - Y)$$

Et pour la vectorisation de la dérivée par rapport à b :

$$\frac{\partial{L}}{\partial{b}} = \frac{1}{m} \sum{(\hat{Y} - Y)}$$

## Propagation avant (Forward) et arrière (Backward) :
Ce sont les prochaines étapes de notre projet. 
La propagation avant est la première étape.
Nous testons nos paramètres avec les données d'entraînement pour observer le coût via la fonction de coût.

### Propagation avant (Forward propagation) :
Nous testons nos paramètres avec les données d'entraînement pour calculer le coût.

In [11]:
def forward_propagation(X, Y, W, b):#ddd
    Z =linear_function(X, W, b)
    Y_hat = sigmoid(Z)
    m = X.shape[1]
    cost = - 1/m * np.sum(Y * np.log(Y_hat) + (1 - Y) * np.log(1 - Y_hat))
    return cost , Y_hat

### Propagation arrière (Backward propagation) :
Ensuite, la propagation arrière consiste à appliquer le principe de la descente de gradient pour minimiser notre fonction de coût. La fonction ci-dessous sera appelée dans une boucle pour mettre à jour les poids (weights) et le biais (bias) à chaque itération.


In [12]:

def back_propagation(W, b, Y, Y_hat, X):
    m = X.shape[1]
    d_W = 1/m *(np.dot(X.T,(Y_hat - Y)))
    d_b = 1/m * np.sum(Y_hat - Y)
    n_W = W - learning_rate * d_W
    n_b = b - learning_rate * d_b
    return n_W, n_b


### Un dernier détail avant de lancer l'algorithme
Le 'y' provenant du jeu de données a une forme (1, m). Pour l'adapter à nos équations et éviter les erreurs de produit matriciel, sa forme doit devenir (m, 1).

In [13]:
y_test = y_test.T
y_train = y_train.T


## Lançons l'algorithme